# 02 · Direct Lake semantic model (as code)

Creates/updates the Power BI semantic model over the gold star schema: tables, relationships,
DAX measures and formatting – idempotent, so it can run after every deployment.
Direct Lake reads the Delta tables in OneLake directly: no import refresh, no duplicate copy.

In [ ]:
semantic_model_name = "sm_football_performance"
lakehouse_name = "lh_football"
overwrite = False

In [ ]:
%pip install -q semantic-link-labs

In [ ]:
import sempy_labs as labs
from sempy_labs.tom import connect_semantic_model
import sempy.fabric as fabric

TABLES = {  # friendly name -> lakehouse table
    "Date": "gold.dim_date", "Competition": "gold.dim_competition_season", "Team": "gold.dim_team",
    "Player": "gold.dim_player", "Position": "gold.dim_position",
    "Match": "gold.dim_match", "Event Type": "gold.dim_event_type",
    "Events": "gold.fact_events", "Shots": "gold.fact_shots", "Passes": "gold.fact_passes",
    "Player Match": "gold.fact_player_match", "Team Match": "gold.fact_team_match",
    "Physical": "gold.fact_player_match_physical",
}
existing = fabric.list_datasets()["Dataset Name"].tolist()
if semantic_model_name not in existing or overwrite:
    labs.directlake.generate_direct_lake_semantic_model(
        dataset=semantic_model_name, tables=TABLES, source=lakehouse_name, source_type="Lakehouse",
        overwrite=overwrite, refresh=False)

In [ ]:
RELATIONSHIPS = [  # (from table, from col, to table, to col, active)
    ("Match", "date_key", "Date", "date_key", True),
    ("Match", "competition_season_key", "Competition", "competition_season_key", True),
    ("Events", "match_key", "Match", "match_key", True), ("Events", "player_key", "Player", "player_key", True),
    ("Events", "team_key", "Team", "team_key", True), ("Events", "event_type_key", "Event Type", "event_type_key", True),
    ("Events", "position_key", "Position", "position_key", True),
    ("Shots", "match_key", "Match", "match_key", True), ("Shots", "player_key", "Player", "player_key", True),
    ("Shots", "team_key", "Team", "team_key", True),
    ("Passes", "match_key", "Match", "match_key", True), ("Passes", "player_key", "Player", "player_key", True),
    ("Passes", "team_key", "Team", "team_key", True),
    ("Player Match", "match_key", "Match", "match_key", True), ("Player Match", "player_key", "Player", "player_key", True),
    ("Player Match", "team_key", "Team", "team_key", True),
    ("Team Match", "match_key", "Match", "match_key", True), ("Team Match", "team_key", "Team", "team_key", True),
    ("Physical", "match_key", "Match", "match_key", True), ("Physical", "player_key", "Player", "player_key", True),
]

MEASURES = {  # table: [(name, DAX, format, folder)]
    "Player Match": [
        ("Minutes", "SUM('Player Match'[minutes_played])", "#,0", "Volume"),
        ("Appearances", "COUNTROWS('Player Match')", "#,0", "Volume"),
        ("Goals", "SUM('Player Match'[goals])", "#,0", "Attacking"),
        ("xG", "SUM('Player Match'[xg])", "0.00", "Attacking"),
        ("npxG", "SUM('Player Match'[npxg])", "0.00", "Attacking"),
        ("xA", "SUM('Player Match'[xa])", "0.00", "Attacking"),
        ("Goals - xG", "[Goals] - [xG]", "+0.00;-0.00;0.00", "Attacking"),
        ("xG per 90", "DIVIDE([xG], [Minutes]) * 90", "0.00", "Per 90"),
        ("npxG + xA per 90", "DIVIDE([npxG] + [xA], [Minutes]) * 90", "0.00", "Per 90"),
        ("Progressive Passes per 90", "DIVIDE(SUM('Player Match'[progressive_passes]), [Minutes]) * 90", "0.0", "Per 90"),
        ("Pressures per 90", "DIVIDE(SUM('Player Match'[pressures]), [Minutes]) * 90", "0.0", "Per 90"),
        ("Pass Completion %", "DIVIDE(SUM('Player Match'[passes_completed]), SUM('Player Match'[passes_attempted]))", "0.0%", "Passing"),
    ],
    "Team Match": [
        ("Team Goals", "SUM('Team Match'[goals_for])", "#,0", "Team"),
        ("Team xG", "SUM('Team Match'[xg])", "0.00", "Team"),
        ("Team xGA", "SUM('Team Match'[xg_against])", "0.00", "Team"),
        ("xG Difference", "[Team xG] - [Team xGA]", "+0.00;-0.00;0.00", "Team"),
        ("Points", "SUM('Team Match'[points])", "#,0", "Team"),
        ("Avg Possession %", "AVERAGE('Team Match'[possession_pct]) / 100", "0.0%", "Team"),
        ("Avg PPDA", "AVERAGE('Team Match'[ppda])", "0.0", "Team"),
    ],
    "Shots": [
        ("Shots", "COUNTROWS(FILTER('Shots', NOT 'Shots'[is_penalty_shootout]))", "#,0", "Shooting"),
        ("Shots on Target %", "DIVIDE(CALCULATE([Shots], 'Shots'[is_on_target]), [Shots])", "0.0%", "Shooting"),
        ("xG per Shot", "DIVIDE(SUM('Shots'[xg]), [Shots])", "0.000", "Shooting"),
    ],
    "Physical": [
        ("Total Distance (km)", "SUM('Physical'[total_distance_m]) / 1000", "0.0", "Physical"),
        ("HSR per 90 (m)", "DIVIDE(SUM('Physical'[hsr_distance_m]), SUM('Physical'[duration_min])) * 90", "#,0", "Physical"),
        ("Pressures per 1km HSR", "DIVIDE(SUM('Player Match'[pressures]), SUM('Physical'[hsr_distance_m]) / 1000)", "0.0", "Physical x Technical"),
    ],
}

with connect_semantic_model(dataset=semantic_model_name, readonly=False) as tom:
    have = {(r.FromTable.Name, r.FromColumn.Name, r.ToTable.Name) for r in tom.model.Relationships}
    for ft, fc, tt, tc, active in RELATIONSHIPS:
        if (ft, fc, tt) not in have:
            tom.add_relationship(from_table=ft, from_column=fc, to_table=tt, to_column=tc,
                                 from_cardinality="Many", to_cardinality="One", is_active=active)
    have_m = {m.Name for m in tom.all_measures()}
    for table, measures in MEASURES.items():
        for name, dax, fmt, folder in measures:
            if name not in have_m:
                tom.add_measure(table_name=table, measure_name=name, expression=dax,
                                format_string=fmt, display_folder=folder)
    tom.mark_as_date_table(table_name="Date", column_name="date")
    # hide surrogate keys from report authors
    for c in tom.all_columns():
        if c.Name.endswith("_key"):
            c.IsHidden = True
print("semantic model ready")

In [ ]:
labs.refresh_semantic_model(dataset=semantic_model_name)